In [ ]:
#CatBoost
from catboost import CatBoostClassifier
import pandas as pd
from preprocessing import train_clean, test_clean

orig_test = pd.read_csv('data/test.csv')

X_train = train_clean.drop(columns = ['Survived', 'PassengerId'], errors = 'ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]

model = CatBoostClassifier(iterations = 200, learning_rate = 0.05, depth = 4, verbose = 0, random_state = 42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': orig_test['PassengerId'],
    'Survived': predictions
})

submission['Survived'] = submission['Survived'].astype(int)

submission.to_csv('submission_baseline.csv', index = False)


In [ ]:
#DNN
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class PoliksDNN(nn.Module):
    def __init__(self, input_dim, hidden_size, activat, drop_rate, batchnorm):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h_size in hidden_size:
            layers.append(nn.Linear(in_dim, h_size))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h_size))
            layers.append(activat)
            if drop_rate > 0:
                layers.append(nn.Dropout(drop_rate))
            in_dim = h_size
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
    
from preprocessing import train_clean, test_clean

orig_test = pd.read_csv('data/test.csv')

X_train = train_clean.drop(columns=['Survived', 'PassengerId'], errors='ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]

hidden_size = [128, 64, 32, 16]
drop_rate = 0.2
batch_size = 64
epochs = 50
lr = 0.01

X_tensor = torch.tensor(X_train.values, dtype = torch.float32)
y_tensor = torch.tensor(y_train.values, dtype = torch.float32).unsqueeze(1)
    
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size = batch_size, shuffle = True)
        
model = PoliksDNN(
    input_dim = X_train.shape[1], 
    hidden_size = hidden_size, 
    activat = nn.GELU(), 
    drop_rate = drop_rate, 
    batchnorm = True
    )

loss_func = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = epochs)
model.train() 
for epoch in range(epochs):
    for x_, y_ in loader:
        optimizer.zero_grad()
        out = model(x_)
        loss = loss_func(out, y_)
        loss.backward()
        optimizer.step()
    scheduler.step()
            
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test.values, dtype = torch.float32)
    test_probs = torch.sigmoid(model(X_test_tensor)).numpy().flatten()
    predictions = (test_probs > 0.5).astype(int)
    y_true = y_tensor.numpy().flatten()

    submission = pd.DataFrame({
    'PassengerId': orig_test['PassengerId'],
    'Survived': predictions
})

submission['Survived'] = submission['Survived'].astype(int)

submission.to_csv('submission_baseline.csv', index = False)

In [ ]:
#Weighted averaging Top Boosters
import pandas as pd
import os
from preprocessing import train_clean, test_clean
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier

X_train = train_clean.drop(columns=['Survived', 'PassengerId'], errors='ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]
orig_test = pd.read_csv('data/test.csv')

best_estimators = {
    'CatBoost': CatBoostClassifier(
        depth = 3, 
        iterations = 150, 
        learning_rate = 0.1,
        verbose = 0,
        random_state = 42
    ),
    'LightGBM': LGBMClassifier(
        learning_rate = 0.1, 
        max_depth = 3, 
        n_estimators = 100,
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        learning_rate = 0.1, 
        max_depth = 3, 
        n_estimators = 100,
        random_state = 42
    )
}

weighted_boosters = VotingClassifier(
    estimators = [
        ('catboost', best_estimators['CatBoost']),
        ('lgbm', best_estimators['LightGBM']),
        ('xgb', best_estimators['XGBoost'])
    ],
    voting = 'soft',
    weights = [3, 2, 2]
)

weighted_boosters.fit(X_train, y_train)
preds = weighted_boosters.predict(X_test)

preds = preds.astype(int)
passenger_id = orig_test['PassengerId'].astype(int)

os.makedirs('submissions', exist_ok = True)
submission = pd.DataFrame({
    'PassengerId': passenger_id,
    'Survived': preds
})

submission.to_csv('submissions/weighted_boosters_submission.csv', index = False, sep = ',', encoding = 'utf-8')

In [ ]:
#Voting hard
import pandas as pd
import os
from preprocessing import train_clean, test_clean
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

X_train = train_clean.drop(columns=['Survived', 'PassengerId'], errors='ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]
orig_test = pd.read_csv('data/test.csv')

cb = CatBoostClassifier(depth=3, iterations=150, learning_rate=0.1, verbose=0, random_state=42)
lgb = LGBMClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42, verbose=-1)
xgb = XGBClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42)
rf = RandomForestClassifier(max_depth=5, n_estimators=200, random_state=42)
lr = LogisticRegression(C=0.1, penalty='l1', solver='liblinear', random_state=42)
svc = SVC(C=1.0, kernel='rbf', random_state=42)

base_estimators = [
    ('catboost', cb),
    ('lgbm', lgb),
    ('xgb', xgb),
    ('rf', rf),
    ('lr', lr),
    ('svc', svc)
]

voting_hard_model = VotingClassifier(
    estimators = base_estimators,
    voting = 'hard'
)

voting_hard_model.fit(X_train, y_train)
preds = voting_hard_model.predict(X_test)

preds = preds.astype(int)
passenger_id = orig_test['PassengerId'].astype(int)

os.makedirs('submissions', exist_ok = True)
submission = pd.DataFrame({
    'PassengerId': passenger_id,
    'Survived': preds
})

submission.to_csv('submissions/voting_hard_submission.csv', index = False, sep = ',', encoding = 'utf-8')

In [ ]:
#Voting soft
import pandas as pd
import os
from preprocessing import train_clean, test_clean
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

X_train = train_clean.drop(columns=['Survived', 'PassengerId'], errors='ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]
orig_test = pd.read_csv('data/test.csv')

cb = CatBoostClassifier(depth=3, iterations=150, learning_rate=0.1, verbose=0, random_state=42)
lgb = LGBMClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42, verbose=-1)
xgb = XGBClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42)
rf = RandomForestClassifier(max_depth=5, n_estimators=200, random_state=42)
lr = LogisticRegression(C=0.1, penalty='l1', solver='liblinear', random_state=42)
svc = SVC(C=1.0, kernel='rbf', probability = True, random_state=42)

base_estimators = [
    ('catboost', cb),
    ('lgbm', lgb),
    ('xgb', xgb),
    ('rf', rf),
    ('lr', lr),
    ('svc', svc)
]

voting_soft_model = VotingClassifier(
    estimators = base_estimators,
    voting = 'soft'
)

voting_soft_model.fit(X_train, y_train)
preds = voting_soft_model.predict(X_test)

preds = preds.astype(int)
passenger_id = orig_test['PassengerId'].astype(int)

os.makedirs('submissions', exist_ok = True)
submission = pd.DataFrame({
    'PassengerId': passenger_id,
    'Survived': preds
})

submission.to_csv('submissions/voting_soft_submission.csv', index = False, sep = ',', encoding = 'utf-8')

In [ ]:
#Stacking Ridge Meta-model
import pandas as pd
import os
from preprocessing import train_clean, test_clean
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import RidgeClassifierCV

X_train = train_clean.drop(columns=['Survived', 'PassengerId'], errors='ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]
orig_test = pd.read_csv('data/test.csv')

cb = CatBoostClassifier(depth=3, iterations=150, learning_rate=0.1, verbose=0, random_state=42)
lgb = LGBMClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42, verbose=-1)
xgb = XGBClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42)
rf = RandomForestClassifier(max_depth=5, n_estimators=200, random_state=42)
lr = LogisticRegression(C=0.1, penalty='l1', solver='liblinear', random_state=42)
svc = SVC(C=1.0, kernel='rbf', probability = True, random_state=42)

base_estimators = [
    ('catboost', cb),
    ('lgbm', lgb),
    ('xgb', xgb),
    ('rf', rf),
    ('lr', lr),
    ('svc', svc)
]

stacking_ridge_model = StackingClassifier(
    estimators = base_estimators,
    final_estimator = RidgeClassifierCV(),
    cv = 5
)

stacking_ridge_model.fit(X_train, y_train)
preds = stacking_ridge_model.predict(X_test)

preds = preds.astype(int)
passenger_id = orig_test['PassengerId'].astype(int)

os.makedirs('submissions', exist_ok = True)
submission = pd.DataFrame({
    'PassengerId': passenger_id,
    'Survived': preds
})

submission.to_csv('submissions/stacking_ridge_meta_model_submission.csv', index = False, sep = ',', encoding = 'utf-8')

In [ ]:
#CatBoost best params
import pandas as pd
import os
from preprocessing import train_clean, test_clean
from catboost import CatBoostClassifier

X_train = train_clean.drop(columns=['Survived', 'PassengerId'], errors='ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]
orig_test = pd.read_csv('data/test.csv')

catboost_model = CatBoostClassifier(depth=3, iterations=150, learning_rate=0.1, verbose=0, random_state=42)

catboost_model.fit(X_train, y_train)
preds = catboost_model.predict(X_test)

preds = preds.astype(int)
passenger_id = orig_test['PassengerId'].astype(int)

os.makedirs('submissions', exist_ok = True)
submission = pd.DataFrame({
    'PassengerId': passenger_id,
    'Survived': preds
})

submission.to_csv('submissions/catboost_model_submission.csv', index = False, sep = ',', encoding = 'utf-8')

In [ ]:
#best votig hard
import os
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from preprocessing import load_data, test_clean, train_clean
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

X_train = train_clean.drop(columns=['Survived'], errors='ignore')
y_train = train_clean['Survived']
X_test = test_clean[X_train.columns]

cb = CatBoostClassifier(
    depth=5, iterations=150, learning_rate=0.03, random_seed=42, verbose=0
)
lgb = LGBMClassifier(
    max_depth=5,
    n_estimators=100,
    learning_rate=0.03,
    random_state=42,
    verbose=-1,
)
xgb = XGBClassifier(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
)

rf = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
)

voting_hard_model = VotingClassifier(
    estimators=[('cat', cb), ('lgb', lgb), ('xgb', xgb), ('rf', rf)],
    voting='hard',
)
voting_hard_model.fit(X_train, y_train)

orig_test = load_data('test.csv')
submission = pd.DataFrame(
    {'PassengerId': orig_test['PassengerId'], 'Survived': voting_hard_model.predict(X_test)}
)

os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/voting_hard_submission.csv', index=False)